# 🌀 OmniGen Image Edit — Free (T4)
T4 Optimized · Gradio UI · Drive I/O · Bulk · Multi-Prompt

> ⚠️ `Runtime → GPU → T4`

In [ ]:
import torch,gc,os
!nvidia-smi
if torch.cuda.is_available(): print(f'GPU:{torch.cuda.get_device_name(0)}|VRAM:{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB')
from google.colab import drive
drive.mount('/content/drive')
def clear_mem():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
def vram():
    if torch.cuda.is_available(): return f'{torch.cuda.memory_allocated()/1024**3:.1f}/{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB'
    return 'N/A'

In [ ]:
!git clone https://github.com/staoxiao/OmniGen.git /content/OmniGen 2>/dev/null; echo 'done'
%cd /content/OmniGen
!pip install -e . -q
!pip install -q gradio
DOUT='/content/drive/MyDrive/OmniGen-Output'; os.makedirs(DOUT,exist_ok=True)
print(f'✅ Output:{DOUT}')

In [ ]:
from OmniGen import OmniGenPipeline
clear_mem()
print('⏳ Loading OmniGen (offload mode for T4)...')
pipe = OmniGenPipeline.from_pretrained('Shitao/OmniGen-v1')
print(f'✅ Loaded! VRAM:{vram()}')
# OmniGen supports offload_model=True in inference calls for T4

In [ ]:
import gradio as gr
import glob,time,tempfile
from PIL import Image
from datetime import datetime
DR='/content/drive/MyDrive'
MAX_RES=768

def smart_resize(img,mx=768):
    w,h=img.size
    if max(w,h)>mx: r=mx/max(w,h); w,h=(int(w*r)//16)*16,(int(h*r)//16)*16; img=img.resize((w,h),Image.LANCZOS)
    return img

def browse_drive(fp):
    if not fp: fp=DR
    if not os.path.isabs(fp): fp=os.path.join(DR,fp)
    if not os.path.exists(fp): return [],f'❌ {fp}'
    exts=('*.png','*.jpg','*.jpeg','*.webp','*.bmp'); files=[]
    for e in exts: files.extend(glob.glob(os.path.join(fp,e))); files.extend(glob.glob(os.path.join(fp,e.upper())))
    files=sorted(set(files))[:100]
    subs=[d for d in sorted(os.listdir(fp)) if os.path.isdir(os.path.join(fp,d)) and not d.startswith('.')]
    return files,f'📂 {fp}\n🖼️ {len(files)}'+(f'\n📁 {", ".join(subs[:20])}' if subs else '')

def load_from_drive(fp):
    if not fp: return None,'❌'
    if not os.path.isabs(fp): fp=os.path.join(DR,fp)
    if not os.path.exists(fp): return None,f'❌ {fp}'
    try: img=Image.open(fp).convert('RGB'); return img,f'✅ {os.path.basename(fp)} ({img.size[0]}x{img.size[1]})'
    except Exception as e: return None,f'❌ {e}'

def save_tmp(img):
    p=os.path.join(tempfile.gettempdir(),f'og_{int(time.time())}.png'); img.save(p); return p

def infer(img_path,prompt,seed,guidance,img_guidance,mx):
    img=Image.open(img_path).convert('RGB')
    img=smart_resize(img,int(mx)); tp2=save_tmp(img)
    h,w=img.size[1],img.size[0]
    h=(h//16)*16; w=(w//16)*16
    images=pipe(
        prompt=f'{prompt} <img><|image_1|></img>',
        input_images=[tp2],
        height=h,width=w,
        guidance_scale=float(guidance),
        img_guidance_scale=float(img_guidance),
        seed=int(seed),
        offload_model=True,
        max_input_image_size=int(mx)
    )
    return images[0]

def edit_single(inp,prompt,seed,guidance,img_guidance,mx,save):
    if inp is None: return None,'❌ Upload!'
    if not prompt.strip(): return None,'❌ Prompt!'
    t=time.time(); clear_mem()
    try:
        img=Image.fromarray(inp).convert('RGB'); tp=save_tmp(img)
        out=infer(tp,prompt,seed,guidance,img_guidance,mx)
        s=f'✅ ({time.time()-t:.1f}s)\n'
        if save:
            p=os.path.join(DOUT,f'omnigen_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png'); out.save(p); s+=f'💾 {p}\n'
        s+=f'VRAM:{vram()}'; return out,s
    except RuntimeError as e:
        if 'out of memory' in str(e).lower(): clear_mem(); return None,'❌ OOM! Kurangi MaxRes.'
        return None,f'❌ {e}'

def bulk_edit(inf,outf,prompt,seed,guidance,img_guidance,mx,progress=gr.Progress()):
    if not prompt.strip() or not inf or not outf: return [],'❌'
    inp=inf if os.path.isabs(inf) else os.path.join(DR,inf)
    outp=outf if os.path.isabs(outf) else os.path.join(DR,outf)
    if not os.path.exists(inp): return [],f'❌ {inp}'
    os.makedirs(outp,exist_ok=True)
    exts=('*.png','*.jpg','*.jpeg','*.webp','*.bmp'); files=[]
    for e in exts: files.extend(glob.glob(os.path.join(inp,e))); files.extend(glob.glob(os.path.join(inp,e.upper())))
    files=sorted(set(files))
    if not files: return [],'❌ No images'
    res=[]; log=f'📦 {len(files)}\n\n'; T=time.time(); ok=0
    for i,fp in enumerate(progress.tqdm(files,desc='Bulk')):
        fn=os.path.basename(fp); log+=f'[{i+1}/{len(files)}] {fn}... '
        try:
            clear_mem(); t=time.time(); out=infer(fp,prompt,seed,guidance,img_guidance,mx)
            out.save(os.path.join(outp,fn)); res.append(os.path.join(outp,fn)); ok+=1
            log+=f'✅ ({time.time()-t:.1f}s)\n'
        except: clear_mem(); log+='❌\n'
    log+=f'\n🎉 {ok}/{len(files)} | ⏱️ {time.time()-T:.0f}s | 💾 {outp}'
    return res,log

def mp_single(inp,pt,seed,guidance,ig,mx,outf,progress=gr.Progress()):
    if inp is None: return [],'❌'
    prompts=[p.strip() for p in pt.strip().split('\n') if p.strip()]
    if not prompts: return [],'❌'
    outp=outf if os.path.isabs(outf) else os.path.join(DR,outf)
    os.makedirs(outp,exist_ok=True)
    img=Image.fromarray(inp).convert('RGB'); tp=save_tmp(img)
    res=[]; log=f'🎨 {len(prompts)}\n\n'; T=time.time(); ok=0
    for i,p in enumerate(progress.tqdm(prompts,desc='MP')):
        log+=f'[{i+1}] "{p[:50]}"... '
        try:
            clear_mem(); t=time.time(); out=infer(tp,p,seed,guidance,ig,mx)
            fn=f'prompt_{i+1}.png'; out.save(os.path.join(outp,fn)); res.append(os.path.join(outp,fn)); ok+=1
            log+=f'✅ ({time.time()-t:.1f}s)\n'
        except: clear_mem(); log+='❌\n'
    log+=f'\n🎉 {ok}/{len(prompts)} | ⏱️ {time.time()-T:.0f}s'
    return res,log

def mp_bulk(inf,pt,seed,guidance,ig,mx,outf,progress=gr.Progress()):
    if not inf: return [],'❌'
    prompts=[p.strip() for p in pt.strip().split('\n') if p.strip()]
    if not prompts: return [],'❌'
    inp=inf if os.path.isabs(inf) else os.path.join(DR,inf)
    outp=outf if os.path.isabs(outf) else os.path.join(DR,outf)
    if not os.path.exists(inp): return [],f'❌ {inp}'
    exts=('*.png','*.jpg','*.jpeg','*.webp','*.bmp'); files=[]
    for e in exts: files.extend(glob.glob(os.path.join(inp,e))); files.extend(glob.glob(os.path.join(inp,e.upper())))
    files=sorted(set(files))
    if not files: return [],'❌'
    total=len(files)*len(prompts)
    log=f'📦 {len(files)}×{len(prompts)}={total}\n\n'; res=[]; T=time.time(); ok=0; n=0
    for pi,p in enumerate(prompts):
        pdir=os.path.join(outp,f'prompt_{pi+1}'); os.makedirs(pdir,exist_ok=True)
        for fi,fp in enumerate(progress.tqdm(files,desc=f'P{pi+1}')):
            fn=os.path.basename(fp); n+=1; log+=f'[{n}/{total}] {fn}|P{pi+1}... '
            try:
                clear_mem(); t=time.time(); out=infer(fp,p,seed,guidance,ig,mx)
                out.save(os.path.join(pdir,fn)); res.append(os.path.join(pdir,fn)); ok+=1
                log+=f'✅ ({time.time()-t:.1f}s)\n'
            except: clear_mem(); log+='❌\n'
    log+=f'\n🎉 {ok}/{total} | ⏱️ {time.time()-T:.0f}s'
    return res,log

THEME=gr.themes.Soft(primary_hue='cyan',secondary_hue='teal',neutral_hue='slate',font=[gr.themes.GoogleFont('Inter'),'sans-serif'])
CSS='.gradio-container{max-width:1200px!important}.mt{text-align:center}.mt h1{background:linear-gradient(135deg,#06b6d4,#14b8a6);-webkit-background-clip:text;-webkit-text-fill-color:transparent;font-size:2em}.sb{font-family:monospace;font-size:.85em}'
with gr.Blocks(theme=THEME,css=CSS,title='OmniGen Free') as demo:
    gr.HTML("<div class='mt'><h1>🌀 OmniGen Image Edit — Free (T4)</h1><p style='color:#888'>T4 · offload · Drive I/O · Bulk · Multi-Prompt</p></div>")
    with gr.Tabs():
        with gr.TabItem('🖼️ Single'):
            with gr.Row():
                with gr.Column():
                    si=gr.Image(type='numpy',height=380)
                    with gr.Accordion('📁 Drive',open=False):
                        sdp=gr.Textbox(label='Path'); slb=gr.Button('📂',size='sm'); sls=gr.Textbox(interactive=False,lines=1)
                    sp=gr.Textbox(label='🎨 Prompt',lines=3,value='Transform into watercolor.')
                    with gr.Row(): ss=gr.Number(label='Seed',value=0,precision=0); sg=gr.Slider(label='Guidance',minimum=1,maximum=5,value=2.5,step=0.5)
                    with gr.Row(): sig=gr.Slider(label='ImgGuidance',minimum=1,maximum=3,value=1.6,step=0.1); smr=gr.Slider(label='MaxRes',minimum=384,maximum=768,value=512,step=64)
                    ssv=gr.Checkbox(label='💾 Save',value=True); sb=gr.Button('🚀 Generate!',variant='primary',size='lg')
                with gr.Column(): so=gr.Image(type='pil',height=380); sot=gr.Textbox(interactive=False,lines=5,elem_classes='sb')
            slb.click(fn=load_from_drive,inputs=[sdp],outputs=[si,sls])
            sb.click(fn=edit_single,inputs=[si,sp,ss,sg,sig,smr,ssv],outputs=[so,sot])
        with gr.TabItem('📦 Bulk'):
            gr.Markdown('### 1 Prompt → folder. Nama file **sama**.')
            with gr.Row(): bf=gr.Textbox(label='📂 In',scale=2); bfp=gr.Button('🔍',scale=1)
            bfi=gr.Textbox(interactive=False,lines=2); bfg=gr.Gallery(columns=6,height=150,object_fit='contain')
            bof=gr.Textbox(label='📂 Out',value='OmniGen-Output/bulk')
            bp=gr.Textbox(label='🎨 Prompt',lines=2,value='Watercolor.')
            with gr.Row(): bsd=gr.Number(label='Seed',value=42,precision=0); bcf=gr.Slider(label='G',minimum=1,maximum=5,value=2.5,step=0.5); big=gr.Slider(label='ImgG',minimum=1,maximum=3,value=1.6,step=0.1); bmr=gr.Slider(label='MaxRes',minimum=384,maximum=768,value=512,step=64)
            bb=gr.Button('📦 Start!',variant='primary',size='lg')
            bl=gr.Textbox(interactive=False,lines=15,elem_classes='sb'); brs=gr.Gallery(columns=5,height=300,object_fit='contain')
            bfp.click(fn=browse_drive,inputs=[bf],outputs=[bfg,bfi])
            bb.click(fn=bulk_edit,inputs=[bf,bof,bp,bsd,bcf,big,bmr],outputs=[brs,bl])
        with gr.TabItem('🎨 Multi-Prompt'):
            with gr.Tabs():
                with gr.TabItem('1×N'):
                    mps_img=gr.Image(type='numpy',height=300)
                    mps_p=gr.Textbox(label='Prompts',lines=6,placeholder='watercolor\nsketch\npop art')
                    with gr.Row(): mps_s=gr.Number(label='Seed',value=42,precision=0); mps_g=gr.Slider(label='G',minimum=1,maximum=5,value=2.5,step=0.5); mps_ig=gr.Slider(label='ImgG',minimum=1,maximum=3,value=1.6,step=0.1); mps_mr=gr.Slider(label='MaxRes',minimum=384,maximum=768,value=512,step=64)
                    mps_o=gr.Textbox(label='📂 Out',value='OmniGen-Output/mp')
                    mps_b=gr.Button('🎨 Run!',variant='primary',size='lg')
                    mps_log=gr.Textbox(interactive=False,lines=10,elem_classes='sb'); mps_gal=gr.Gallery(columns=4,height=400,object_fit='contain')
                    mps_b.click(fn=mp_single,inputs=[mps_img,mps_p,mps_s,mps_g,mps_ig,mps_mr,mps_o],outputs=[mps_gal,mps_log])
                with gr.TabItem('N×N'):
                    with gr.Row(): mpb_f=gr.Textbox(label='📂 In',scale=2); mpb_prev=gr.Button('🔍',scale=1)
                    mpb_info=gr.Textbox(interactive=False,lines=2); mpb_gin=gr.Gallery(columns=6,height=150,object_fit='contain')
                    mpb_p=gr.Textbox(label='Prompts',lines=6)
                    with gr.Row(): mpb_s=gr.Number(label='Seed',value=42,precision=0); mpb_g=gr.Slider(label='G',minimum=1,maximum=5,value=2.5,step=0.5); mpb_ig=gr.Slider(label='ImgG',minimum=1,maximum=3,value=1.6,step=0.1); mpb_mr=gr.Slider(label='MaxRes',minimum=384,maximum=768,value=512,step=64)
                    mpb_o=gr.Textbox(label='📂 Out',value='OmniGen-Output/mp_bulk')
                    mpb_b=gr.Button('📦🎨 Run!',variant='primary',size='lg')
                    mpb_log=gr.Textbox(interactive=False,lines=15,elem_classes='sb'); mpb_gal=gr.Gallery(columns=5,height=400,object_fit='contain')
                    mpb_prev.click(fn=browse_drive,inputs=[mpb_f],outputs=[mpb_gin,mpb_info])
                    mpb_b.click(fn=mp_bulk,inputs=[mpb_f,mpb_p,mpb_s,mpb_g,mpb_ig,mpb_mr,mpb_o],outputs=[mpb_gal,mpb_log])
        with gr.TabItem('📁 Drive'):
            with gr.Row(): gf=gr.Textbox(value='',scale=3); gb=gr.Button('🔍',variant='primary',scale=1)
            gi=gr.Textbox(interactive=False,lines=3); gg=gr.Gallery(columns=5,height=450,object_fit='contain')
            gb.click(fn=browse_drive,inputs=[gf],outputs=[gg,gi])
    gr.Markdown('---\n🌀 OmniGen-v1')
demo.launch(share=True,debug=True)